In [ ]:
# =============================================================================
# jobs/33_build_address_points.py
# Address points for the exposure and risk grid: every England UPRN with its share of the Census 2021 people, its deprivation, and the
# flood band at its position (plan: docs/EXPOSURE_AND_RISK_GRID_PLAN.md). Address-level, so the table is in the RESTRICTED schema.
#
# INPUTS  ONS National Statistics UPRN Lookup (NSUL), governed volume: UPRN, grid reference, output area, LSOA, local authority, county,
#         ward, constituency, police force, region.
#         config.TBL_CENSUS_OA and config.TBL_IMD_LSOA (job 29), the output area centres (ONS boundary file, columns only, no geometry),
#         and the rasters of job 30 (config.HAZARD_RASTER_DIR).
# OUTPUT  config.TBL_ADDRESS_POINTS   uprn, easting, northing, cell_key, oa, lsoa, lad, ctyua, ward, pcon, pfa, rgn,
#                                     people, households, over65 (this point's equal share of its output area),
#                                     imd_score, imd_decile, tile, point_source (nsul or oa_centroid),
#                                     rofrs_rank (rivers and sea, 0 none to 4 High), fz_rank (0 none, 1 FZ2, 2 FZ3),
#                                     fz_sea_rank (coastal), fz_river_rank (fluvial), fz_river_and_sea_rank (mapped together)
# Coastal, fluvial and surface water are different hazards: no surface water here, and the Flood Zone ranks are kept in separate columns.
#
# HOW  Points per output area are counted once over England, so each 50 km tile can then be built on its own (in parallel with
# workers > 1). Output areas with Census people but no UPRN get one point at their centre carrying all of them, so totals are conserved.
# NSUL holds homes and businesses alike, so people are shared a little too widely in town centres: AddressBase Premium would fix that.
#
# WHAT TO CHECK (printed): rows dropped for no grid reference; the country split; people in the table against the Census total (should match);
# people by rivers and sea band; points per tile (a tile far from the rest, or with none, is worth a look).
#
# RESUME  A tile is finished when its rows are in the table. With resume = yes a re-run skips finished tiles.
# Widgets: tiles, resume, workers, nsul_dir, oa_file, raster_dir, england_only.
# =============================================================================

import os
import sys

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

# shapely is not on every cluster. If it has to be installed the notebook restarts Python: run all again once.
try:
    import shapely  # noqa: F401
except ImportError:
    get_ipython().run_line_magic("pip", "install -q shapely")
    dbutils.library.restartPython()

import importlib
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

import config
import utils.hazard_raster as _hz
import utils.address_points as _ap
importlib.reload(_hz)
importlib.reload(_ap)
from utils.address_points import HAZARD_SOURCES, NSUL_COLUMNS, build_points, centroid_points, clean_nsul, oa_point_counts, oa_shares
from utils.hazard_raster import tile_bounds, tile_code_at
from utils.helpers import get_spark

spark = get_spark()

ONS = "/Volumes/prd_dash_bronze/ons_open_geography_portal_unrestricted"
dbutils.widgets.text("tiles", "", "Only these tiles, for a trial (comma separated, e.g. SE50,TQ05). Blank is all")
dbutils.widgets.dropdown("resume", "yes", ["yes", "no"], "Skip tiles already in the table")
dbutils.widgets.text("workers", "1", "Tiles to run at once (8 on a 16 core node)")
dbutils.widgets.text("nsul_dir", f"{ONS}/national_statistics_uprn_lookup/format_PARQUET_national_statistics_uprn_lookup/LATEST_national_statistics_uprn_lookup/", "NSUL folder")
dbutils.widgets.text("oa_file", f"{ONS}/output_areas_2021_ew_bfe/format_GEOPARQUET_output_areas_2021_ew_bfe/LATEST_output_areas_2021_ew_bfe", "Output areas file (for centres)")
dbutils.widgets.text("raster_dir", config.HAZARD_RASTER_DIR, "Folder of job 30 rasters")
dbutils.widgets.dropdown("england_only", "yes", ["yes", "no"], "England only")

ONLY = [t.strip().upper() for t in dbutils.widgets.get("tiles").split(",") if t.strip()]
RESUME = dbutils.widgets.get("resume") == "yes"
WORKERS = max(1, int(dbutils.widgets.get("workers") or 1))
NSUL_DIR, OA_FILE = dbutils.widgets.get("nsul_dir"), dbutils.widgets.get("oa_file")
RASTER_DIR = dbutils.widgets.get("raster_dir")
ENGLAND_ONLY = dbutils.widgets.get("england_only") == "yes"
OUT = config.TBL_ADDRESS_POINTS
print(f"Workers {WORKERS}. Resume {RESUME}. Trial tiles: {ONLY or 'none (all)'}. Rasters in {RASTER_DIR}")
missing = [s for s in HAZARD_SOURCES if not os.path.isdir(os.path.join(RASTER_DIR, s))]
if missing:
    print("WARNING: no raster folder for", missing, "- those bands will be 0 everywhere. Run job 30 first.")


In [ ]:
# =============================================================================
# 1. NSUL: WHAT IS IN IT, AND HOW MANY POINTS EACH OUTPUT AREA HAS
# =============================================================================

wanted = list(NSUL_COLUMNS)
nsul = spark.read.parquet(NSUL_DIR).select(*wanted)
for c in ("GRIDGB1E", "GRIDGB1N"):
    nsul = nsul.withColumn(c, F.col(c).cast("double"))

by_country = nsul.groupBy("CTRY25CD").count().toPandas().sort_values("count", ascending=False)
print("NSUL rows by country:")
display(by_country.astype(str))
placed = nsul.filter((F.col("GRIDGB1E") > 0) & (F.col("GRIDGB1N") > 0))
if ENGLAND_ONLY:
    placed = placed.filter(F.col("CTRY25CD") == "E92000001")
n_all, n_placed = nsul.count(), placed.count()
print(f"Rows {n_all:,}; England rows with a grid reference {n_placed:,}; dropped (no grid reference or not England) {n_all - n_placed:,}.")
dup = placed.groupBy("UPRN").count().filter("count > 1").count()
print(f"Duplicate UPRNs: {dup:,} (should be 0).")

oa_counts = placed.groupBy("OA21CD").count().withColumnRenamed("OA21CD", "oa").withColumnRenamed("count", "n_points").toPandas()
print(f"Output areas with at least one point: {len(oa_counts):,}. Points per output area: median {oa_counts['n_points'].median():.0f}, max {oa_counts['n_points'].max():,}.")


In [ ]:
# =============================================================================
# 2. CENSUS, DEPRIVATION, SHARES, AND OUTPUT AREAS WITH NO POINTS
# =============================================================================

census = spark.table(config.TBL_CENSUS_OA).toPandas()
imd = spark.table(config.TBL_IMD_LSOA).toPandas()
shares = oa_shares(census, oa_counts)
print(f"Census output areas {len(census):,}; with points {len(shares):,}; population in the Census table {int(census['population'].sum()):,}.")

# the output area centres are only needed for an output area that has people but no UPRN, so read the file only if there is one
n_without = len(set(census["oa"]) - set(oa_counts["oa"]))
if n_without:
    centres = (spark.read.parquet(OA_FILE).select("OA21CD", "LSOA21CD", "BNG_E", "BNG_N").toPandas()
               .rename(columns={"OA21CD": "oa", "LSOA21CD": "lsoa", "BNG_E": "easting", "BNG_N": "northing"}))
else:
    print("Every Census output area has at least one UPRN: no centre points are needed.")
    centres = pd.DataFrame({"oa": [], "lsoa": [], "easting": [], "northing": []})
no_point_pts, no_point_report = centroid_points(census, oa_counts, centres, imd, RASTER_DIR)
print("Output areas with people but no UPRN:", no_point_report)

# tiles that hold points, found once
tiles_df = (placed.select((F.floor(F.col("GRIDGB1E") / 50000)).alias("tx"), (F.floor(F.col("GRIDGB1N") / 50000)).alias("ty"))
            .groupBy("tx", "ty").count().toPandas())
tiles_df["tile"] = tile_code_at((tiles_df["tx"] * 50000 + 1.0).to_numpy(), (tiles_df["ty"] * 50000 + 1.0).to_numpy())
tiles_df = tiles_df[tiles_df["tile"] != ""].sort_values("tile").reset_index(drop=True)
if ONLY:
    tiles_df = tiles_df[tiles_df["tile"].isin(ONLY)]
print(f"{len(tiles_df)} tiles hold points; the biggest has {int(tiles_df['count'].max()):,}.")


In [ ]:
# =============================================================================
# 3. BUILD EACH TILE AND APPEND IT
# =============================================================================

done_tiles = set()
table_exists = spark.catalog.tableExists(OUT)
if table_exists and RESUME:
    done_tiles = set(spark.table(OUT).select("tile").distinct().toPandas()["tile"])
elif table_exists and not ONLY:
    spark.sql(f"DROP TABLE {OUT}")
    table_exists = False
print(f"Tiles already in the table: {len(done_tiles)}")

log = []


from pyspark.sql.types import ByteType, DoubleType, IntegerType, LongType, StringType, StructField, StructType

STR = ["uprn", "oa", "lsoa", "lad", "ctyua", "ward", "pcon", "pfa", "rgn", "tile", "point_source"]
SCHEMA = StructType(
    [StructField("uprn", StringType()), StructField("easting", DoubleType()), StructField("northing", DoubleType()),
     StructField("cell_key", LongType())]
    + [StructField(c, StringType()) for c in ("oa", "lsoa", "lad", "ctyua", "ward", "pcon", "pfa", "rgn")]
    + [StructField(c, DoubleType()) for c in ("people", "households", "over65", "imd_score")]
    + [StructField("imd_decile", IntegerType()), StructField("tile", StringType()), StructField("point_source", StringType())]
    + [StructField(c, ByteType()) for c in HAZARD_SOURCES.values()]
)


def append(pdf):
    """Write a frame with an explicit schema: the centre points have empty text columns that Spark cannot infer."""
    out = pdf.reindex(columns=[f.name for f in SCHEMA]).copy()
    for c in STR:
        out[c] = out[c].astype("object").where(out[c].notna(), None)
    out["imd_decile"] = pd.to_numeric(out["imd_decile"], errors="coerce").round().astype("Int64")
    for c in HAZARD_SOURCES.values():
        out[c] = out[c].astype("int8")
    spark.createDataFrame(out, schema=SCHEMA).write.format("delta").mode("append").saveAsTable(OUT)


def do_tile(row, i, n):
    tile = row["tile"]
    if tile in done_tiles:
        log.append({"tile": tile, "status": "skipped (done)"})
        return
    t0 = time.time()
    try:
        x0, y0 = int(row["tx"]) * 50000, int(row["ty"]) * 50000
        sub = placed.filter((F.col("GRIDGB1E") >= x0) & (F.col("GRIDGB1E") < x0 + 50000) & (F.col("GRIDGB1N") >= y0) & (F.col("GRIDGB1N") < y0 + 50000))
        raw = sub.toPandas()
        pts, _ = clean_nsul(raw, england_only=ENGLAND_ONLY)
        built = build_points(pts, shares, imd, RASTER_DIR)
        if ONLY and table_exists:
            spark.sql(f"DELETE FROM {OUT} WHERE tile = '{tile}'")
        append(built)
        log.append({"tile": tile, "status": "done", "points": len(built), "people": float(built["people"].sum()),
                    "in_rofrs_high": int((built["rofrs_rank"] == 4).sum()), "seconds": round(time.time() - t0, 1)})
        print(f"[{i}/{n}] {tile}: {len(built):,} points, {time.time() - t0:.0f}s")
    except Exception as e:
        log.append({"tile": tile, "status": "FAILED: " + str(e)[:150]})
        print(f"[{i}/{n}] {tile}: FAILED {str(e)[:150]}")


rows = [(r, i, len(tiles_df)) for i, (_, r) in enumerate(tiles_df.iterrows(), 1)]
if WORKERS == 1:
    for w in rows:
        do_tile(*w)
else:
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        for f in as_completed([pool.submit(do_tile, *w) for w in rows]):
            f.result()

# the output areas with no UPRN, once, unless a trial
if not ONLY and len(no_point_pts) and (not RESUME or "oa_centroid" not in set(spark.table(OUT).select("point_source").distinct().toPandas()["point_source"])):
    no_point_pts["tile"] = tile_code_at(no_point_pts["easting"].to_numpy(), no_point_pts["northing"].to_numpy())
    append(no_point_pts)
    print(f"Added {len(no_point_pts):,} output area centre points.")


In [ ]:
# =============================================================================
# 4. CHECKS
# =============================================================================

log_df = pd.DataFrame(log)
print("Tiles by status:")
display(log_df.groupby("status").size().rename("tiles").reset_index().astype(str))
failed = log_df[log_df["status"].str.startswith("FAILED")]
if len(failed):
    print("Failed tiles (re-run with resume = yes to retry only these):")
    display(failed.astype(str))

t = spark.table(OUT)
tot = t.agg(F.count("*").alias("points"), F.sum("people").alias("people"), F.sum("households").alias("households"), F.sum("over65").alias("over65")).toPandas().iloc[0]
print(f"\nIn the table: {int(tot['points']):,} points, {tot['people']:,.0f} people, {tot['households']:,.0f} households, {tot['over65']:,.0f} aged 65 or over.")
census_people = float(census["population"].sum())
print(f"Census total {census_people:,.0f}. Difference {tot['people'] - census_people:,.0f} "
      f"(0 for a full run; after a trial it is the people outside the trial tiles; people lost where an output area has no centre: {no_point_report['people_lost']:,.0f}).")

print("\nPeople by rivers and sea band (rank 0 none, 1 Very Low, 2 Low, 3 Medium, 4 High). Coastal, fluvial and surface water are not merged here:")
display(t.groupBy("rofrs_rank").agg(F.count("*").alias("points"), F.round(F.sum("people")).alias("people")).orderBy("rofrs_rank").toPandas().astype(str))
print("People by Flood Zone, all, and by mapping source (sea = coastal, river = fluvial):")
fz = [t.groupBy(F.lit(c).alias("layer"), F.col(c).alias("rank")).agg(F.round(F.sum("people")).alias("people"))
      for c in ("fz_rank", "fz_sea_rank", "fz_river_rank", "fz_river_and_sea_rank")]
from functools import reduce
display(reduce(lambda a, b: a.unionByName(b), fz).orderBy("layer", "rank").toPandas().astype(str))
print("Points per point source:")
display(t.groupBy("point_source").count().toPandas().astype(str))

spark.sql(f"""COMMENT ON TABLE {OUT} IS 'One row per England UPRN (ONS NSUL) plus one centre point per output area with people but no UPRN. People, households and over-65s are the point''s equal share of its Census 2021 output area; deprivation is the IoD2025 LSOA decile; rofrs_rank is the rivers and sea band and the fz columns the Flood Zone rank at the point, read from the job 30 rasters (within 7 m of a polygon edge). Restricted: address-level. Job 33.'""")


In [ ]:
# Exit is in its own cell: in an interactive run Databricks replaces a cell's output with the exit message,
# so the checks above would be hidden if this line shared their cell.
dbutils.notebook.exit("success" if not len(failed) else "finished with failed tiles")
